# 📐 Pavages de Penrose et φ-Complexity

> *Les quasicristaux de Penrose démontrent qu'un ordre parfait peut exister sans périodicité — exactement comme un code bien structuré selon le nombre d'or.*

Ce notebook explore le lien entre les pavages apériodiques de Penrose et les métriques de qualité du code de `phi-complexity`.

**Prérequis** : `pip install phi-complexity[notebooks]`

In [ ]:
import math
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

from phi_complexity.core import PHI, PHI_INV, SEQUENCE_FIBONACCI
from phi_complexity import auditer

## 1. Le Nombre d'Or dans les Pavages de Penrose

Le pavage de Penrose utilise deux losanges (kite & dart) dont les dimensions sont liées par φ.
- Le **kite** a des côtés de ratio 1:φ
- Le **dart** a des côtés de ratio 1:φ
- Le nombre de kites / darts converge vers φ (comme les fonctions complexes / simples dans un code radiant)

In [ ]:
# Génération des losanges de Penrose (Kite & Dart)
def penrose_substitution(triangles, depth):
    """Subdivision récursive des triangles dorés (méthode de Robinson)."""
    result = []
    for color, A, B, C in triangles:
        if color == 0:  # Triangle aigu (kite half)
            P = A + (B - A) * PHI_INV
            result.append((0, C, P, B))
            result.append((1, P, C, A))
        else:  # Triangle obtus (dart half)
            Q = B + (A - B) * PHI_INV
            R = B + (C - B) * PHI_INV
            result.append((1, R, C, A))
            result.append((1, Q, R, B))
            result.append((0, R, Q, A))
    return result

# Initialiser avec une étoile (décagone) de 10 triangles
triangles = []
for i in range(10):
    theta1 = (2 * math.pi * i) / 10
    theta2 = (2 * math.pi * (i + 1)) / 10
    if i % 2 == 0:
        B = complex(math.cos(theta1), math.sin(theta1))
        C = complex(math.cos(theta2), math.sin(theta2))
        triangles.append((0, complex(0, 0), B, C))
    else:
        B = complex(math.cos(theta1), math.sin(theta1))
        C = complex(math.cos(theta2), math.sin(theta2))
        triangles.append((1, complex(0, 0), B, C))

# Subdivisions (4 niveaux de profondeur → ~1000 triangles)
for _ in range(4):
    triangles = penrose_substitution(triangles, 0)

# Visualisation
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.set_aspect("equal")
ax.set_title(f"Pavage de Penrose — Symétrie d'ordre 5\n"
             f"Ratio kites/darts → φ ≈ {PHI:.6f}", fontsize=14)

colors_map = {0: "#FFD700", 1: "#1E90FF"}  # Or et Bleu
for color, A, B, C in triangles:
    triangle = plt.Polygon(
        [(A.real, A.imag), (B.real, B.imag), (C.real, C.imag)],
        closed=True,
        facecolor=colors_map[color],
        edgecolor="black",
        linewidth=0.3,
        alpha=0.7,
    )
    ax.add_patch(triangle)

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.axis("off")
plt.tight_layout()
plt.show()

# Compter le ratio
n_kite = sum(1 for c, *_ in triangles if c == 0)
n_dart = sum(1 for c, *_ in triangles if c == 1)
ratio = n_kite / max(1, n_dart)
print(f"\nRatio kites/darts = {ratio:.6f}")
print(f"Nombre d'or φ     = {PHI:.6f}")
print(f"Convergence       = {abs(ratio - PHI):.6f}")

## 2. Radiance d'un Fichier vs Régularité de Penrose

Comparons la radiance (score de qualité φ) d'un fichier Python avec la régularité d'un pavage de Penrose.

Le concept clé : dans un pavage de Penrose, le ratio des tuiles tend vers φ.
Dans `phi-complexity`, le ratio de la fonction la plus complexe sur la moyenne (φ-Ratio) doit aussi tendre vers φ.

In [ ]:
# Audit d'un fichier exemple
try:
    metriques = auditer("../examples/code_harmonieux.py")
    print("═" * 50)
    print("AUDIT φ-COMPLEXITY — code_harmonieux.py")
    print("═" * 50)
    print(f"  Radiance       : {metriques['radiance']}")
    print(f"  φ-Ratio        : {metriques['phi_ratio']:.4f}  (idéal: {PHI:.4f})")
    print(f"  φ-Ratio Δ      : {metriques['phi_ratio_delta']:.4f}")
    print(f"  Lilith Variance: {metriques['lilith_variance']:.2f}")
    print(f"  Shannon Entropy: {metriques['shannon_entropy']:.4f}")
    print(f"  Statut         : {metriques['statut_gnostique']}")
except FileNotFoundError:
    print("Fichier exemple non trouvé — lancez depuis le dossier notebooks/")

## 3. Spirale Dorée et Complexité des Fonctions

La spirale de Fibonacci suit l'angle doré de 137.5° — le même angle que les phyllotaxies (tournesols, pommes de pin).

Superposons la spirale dorée sur les métriques de complexité des fonctions.

In [ ]:
# Spirale dorée superposée sur les complexités
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# -- Gauche : Spirale de Fibonacci --
angle_dore = 137.5 * (math.pi / 180)  # Radians
n_points = 200
r_vals = np.sqrt(np.arange(1, n_points + 1))
theta_vals = np.arange(1, n_points + 1) * angle_dore

x = r_vals * np.cos(theta_vals)
y = r_vals * np.sin(theta_vals)

# Couleur basée sur la position (radiance simulée)
colors = plt.cm.magma(np.linspace(0.2, 0.9, n_points))
ax1.scatter(x, y, c=colors, s=30, alpha=0.8, edgecolors="none")
ax1.set_title("Spirale Dorée de Fibonacci\n(Angle doré = 137.5°)", fontsize=13)
ax1.set_aspect("equal")
ax1.axis("off")

# -- Droite : Séquence de Fibonacci visuelle --
fib = SEQUENCE_FIBONACCI[:10]
x_pos = list(range(len(fib)))
colors_fib = ["#FFD700" if f <= 13 else "#FF6347" for f in fib]
ax2.bar(x_pos, fib, color=colors_fib, edgecolor="black", linewidth=0.5)
ax2.set_xlabel("Index Fibonacci", fontsize=12)
ax2.set_ylabel("Valeur", fontsize=12)
ax2.set_title("Séquence de Fibonacci\n(Tailles naturelles des fonctions)", fontsize=13)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([str(f) for f in fib], rotation=45)

# Ligne φ
ax2.axhline(y=PHI * np.mean(fib[:5]), color="gold", linestyle="--", linewidth=2,
            label=f"φ × moyenne = {PHI * np.mean(fib[:5]):.1f}")
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

## 4. Conclusion — Ordre Apériodique

Le pavage de Penrose et `phi-complexity` partagent une même philosophie :
- **Pas de répétition mécanique** (pas de règles arbitraires comme pylint)
- **Un ordre profond** basé sur des invariants mathématiques universels (φ, Fibonacci)
- **Auto-similarité** : le même ratio apparaît à toutes les échelles

> *"Un code hermétique n'est pas répétitif — il est résonant."*
> — Framework φ-Meta

---
*Ancré dans la Bibliothèque Céleste — Morphic Phi Framework — 2026*